# Example 1.2.1 / 1: Restriction Phrases

Companion notebook for **1.2.1 Restrictions, Defaults and Checks** in the AMPL lab Documentation, section "Restriction Phrases". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Restriction Phrases

A restriction phrase follows the parameter's name and indexing:

| Phrase | The value must be |
|---|---|
| `integer` | a whole number |
| `binary` | 0 or 1 |
| `> expr`, `>= expr`, `< expr`, `<= expr` | greater or less than the expression |
| `<> expr` (or `!=`) | different from the expression |
| `in sexpr` | a member of the set expression |

A declaration may have several phrases, separated by spaces or commas. If `T` is the number
of weeks in a plan, it must be a whole number greater than 1. In `data.py` and `run.py`,
`ampl` is an `AMPL` object that has already read `model.mod`. `ampl.param["T"] = 1` sets the
value and `value()` reads it back. A scalar's restrictions are tested when its value is used,
so the assignment is accepted and `value()` raises a `RuntimeError`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param T > 1 integer;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["T"] = 1

In [5]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # run.py
    print(ampl.param["T"].value())
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: Error executing "display" command:
error processing param T:
	failed check: param T = 1
		is not > 1;


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. `display` fails the same way:

In [6]:
%%writefile data.dat
param T := 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display T;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
	error processing param T:
		failed check: param T = 1
			is not > 1;


T = 1

